In [1]:
# Install any extra libraries (usually already available in Colab)
!pip install scikit-learn joblib -q

# Core libraries
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

import joblib
import os


In [2]:
# Replace with the correct filename/path if needed
df = pd.read_csv("/content/asia_snapshot.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Shape: (2003, 17)
Columns: ['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'true_track', 'vertical_rate', 'sensors', 'geo_altitude', 'squawk', 'spi', 'position_source']


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,39de4e,TVF32DD,France,1.755600e+09,1755600237,27.5943,40.9151,3535.68,False,155.44,83.16,-13.98,NaN,3718.56,674.0,False,0
1,aa9321,UAL803,United States,1.755600e+09,1755600238,140.1061,35.9174,2202.18,False,142.66,182.69,-2.93,NaN,2392.68,3242.0,False,0
2,801645,AIC402,India,1.755600e+09,1755600239,72.8513,20.1017,8892.54,False,196.29,7.83,3.25,NaN,9403.08,NaN,False,0
3,80160b,IGO2575,India,1.755600e+09,1755600237,77.2031,26.1268,10972.80,False,238.01,7.20,0.00,NaN,11727.18,NaN,False,0
4,801615,AXB2980,India,1.755600e+09,1755600225,77.9805,13.4514,2628.90,False,103.23,192.37,-3.90,NaN,2598.42,NaN,False,0


In [3]:
# Keep only rows with valid positions (lat/lon are essential)
df_clean = df.dropna(subset=["latitude", "longitude"]).copy()

# Convert numeric columns we will use
num_cols = ["velocity", "baro_altitude", "vertical_rate", "true_track"]
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Remove rows where these features are missing
df_clean = df_clean.dropna(subset=num_cols).copy()

print("After cleaning:", df_clean.shape)
df_clean[num_cols].describe()


After cleaning: (1881, 17)


,velocity,baro_altitude,vertical_rate,true_track
count,1881.000000,1881.000000,1881.000000,1881.000000
mean,197.647996,7656.730750,0.094301,194.915375
std,63.325608,3936.134142,5.108149,101.906534
min,2.620000,22.860000,-23.730000,0.120000
25%,157.350000,4114.800000,-0.330000,113.460000
50%,224.360000,9410.700000,0.000000,193.940000
75%,240.330000,10972.800000,0.330000,292.140000
max,623.920000,21061.680000,24.380000,359.730000


In [4]:
# Features we will feed to Isolation Forest
feature_cols = ["velocity", "baro_altitude", "vertical_rate", "true_track"]

X = df_clean[feature_cols].values
print("Feature matrix shape:", X.shape)

# Scaling: zero mean, unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]


Feature matrix shape: (1881, 4)


array([[-0.66670058, -1.04725766, -2.75599696, -1.09693744],
       [-0.86856834, -1.38613194, -0.59221159, -0.11999845],
       [-0.02145036,  0.31404872,  0.61794168, -1.83634078],
       [ 0.63754204,  0.84269259, -0.01846578, -1.84252456],
       [-1.49138882, -1.27769217, -0.78215474, -0.02498418]])

In [5]:
# Approximate fraction of anomalies in the dataset (tune if needed)
contamination_rate = 0.10  # 10%

iso = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination=contamination_rate,
    random_state=42,
    bootstrap=False,
)

iso.fit(X_scaled)


IsolationForest(contamination=0.1, n_estimators=200, random_state=42)

In [6]:
# 1 = normal, -1 = anomaly (Isolation Forest convention)
labels = iso.predict(X_scaled)
scores = iso.score_samples(X_scaled)  # lower score = more anomalous

df_clean["if_label"] = labels
df_clean["if_score"] = scores

print("Label counts (1 normal, -1 anomaly):")
print(df_clean["if_label"].value_counts())

df_clean[["if_score"]].describe()


Label counts (1 normal, -1 anomaly):
if_label
 1    1693
-1     188
Name: count, dtype: int64


,if_score
count,1881.000000
mean,-0.460851
std,0.061216
min,-0.663018
25%,-0.507192
50%,-0.460883
75%,-0.400096
max,-0.372793


In [7]:
# Subset of points labelled as anomalies
anoms = df_clean[df_clean["if_label"] == -1].copy()

print("Number of anomalies:", len(anoms))
anoms[["icao24", "callsign", "origin_country",
       "velocity", "baro_altitude", "vertical_rate",
       "true_track", "if_score"]].head(10)


Number of anomalies: 188


,icao24,callsign,origin_country,velocity,baro_altitude,vertical_rate,true_track,if_score
0,39de4e,TVF32DD,France,155.44,3535.68,-13.98,83.16,-0.563857
22,80167e,AIC2620,India,188.29,4427.22,14.31,202.15,-0.559295
41,489511,SPFIS,Poland,41.18,640.08,2.28,77.01,-0.572525
50,44020a,AUA810Y,Austria,207.90,6248.40,12.35,343.76,-0.563239
59,4880d4,SPAGW,Poland,37.22,304.80,3.90,139.48,-0.578011
64,8014dc,IGO764,India,216.76,8046.72,10.73,14.99,-0.557506
79,469c25,SXGAE,Greece,28.85,762.00,-0.33,111.99,-0.559410
95,505cea,SEU7834,Slovakia,220.02,8884.92,-12.03,352.48,-0.571487
101,511050,ESANS,Estonia,42.07,419.10,0.00,258.00,-0.549405
102,1427dd,TRH9111,Russian Federation,28.08,2758.44,-0.33,261.57,-0.566379


In [8]:
# Create models directory
os.makedirs("models", exist_ok=True)

joblib.dump(iso, "models/adsb_iforest.pkl")
joblib.dump(scaler, "models/adsb_if_scaler.pkl")

print("Saved: models/adsb_iforest.pkl and models/adsb_if_scaler.pkl")


Saved: models/adsb_iforest.pkl and models/adsb_if_scaler.pkl


In [14]:
from google.colab import files

# First: Isolation Forest model
files.download("models/adsb_iforest.pkl")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
files.download("models/adsb_iforest.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import os

print(os.listdir())
print("models dir:", os.listdir("models"))


['.config', 'asia_snapshot.csv', 'models', 'drive', 'sample_data']
models dir: ['adsb_iforest.pkl', 'adsb_if_scaler.pkl']


In [15]:
import os

print(os.path.exists("models/adsb_iforest.pkl"))
print(os.path.exists("models/adsb_if_scaler.pkl"))


True
True
